### Import libraries

In [1]:
import pandas as pd
from darts.timeseries import TimeSeries
from darts.utils.timeseries_generation import datetime_attribute_timeseries
from darts.dataprocessing.transformers import Scaler
from darts.models import TFTModel
from darts.dataprocessing.transformers import StaticCovariatesTransformer
import numpy as np
import torch
import matplotlib.pyplot as plt

c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md


### Read the data

In [2]:
pandas_df = pd.read_csv(r"top_10_series_data.csv",index_col = ['MONTH_OF_SALE'],parse_dates=True)

In [3]:
pandas_df["DEALER_CODE"] = pandas_df["DEALER_CODE"].astype('str')

In [4]:
pandas_df.index.min()

Timestamp('2023-04-01 00:00:00')

In [5]:
pandas_df["DEALER_CITY"].dtypes

dtype('O')

In [6]:
#Extracting type of columns according to the datatypes
# 1. Targets/Metrics (The numbers we want to predict)
target_cols = pandas_df.select_dtypes(include=['number']).columns.tolist()
target_cols.append('PARENT_DEALER_CODE_MODEL_FAMILY')

# 2. Time Dimension
time_cols = pandas_df.select_dtypes(include=['datetime', 'datetime64']).columns.tolist()

# 3. Static/Categorical Covariates (The identifiers)
# We exclude numbers and dates to find the "ID" strings
static_cols = pandas_df.select_dtypes(exclude=['number', 'datetime', 'datetime64']).columns.tolist()

print(f"Targets: {target_cols}")
print(f"Time Column: {time_cols}")
print(f"Static Identifiers: {static_cols}")

Targets: ['NUM_FESTIVE_DAYS_MONTH', 'FESTIVE_PHASE_I', 'FESTIVE_PHASE_II', 'FESTIVE_PHASE_III', 'TOTAL_DAYS_FESTIVE_PHASE_I', 'TOTAL_DAYS_FESTIVE_PHASE_II', 'TOTAL_DAYS_FESTIVE_PHASE_III', 'TOTAL_DAYS_PITRU_PAKSH', 'PROP_FESTIVE_PHASE_I', 'PROP_EVENT_FESTIVE_PHASE_I', 'PROP_FESTIVE_PHASE_II', 'PROP_EVENT_FESTIVE_PHASE_II', 'PROP_FESTIVE_PHASE_III', 'PROP_EVENT_FESTIVE_PHASE_III', 'PROP_PITRU_PAKSH', 'PROP_EVENT_PITRU_PAKSH', 'NET_SALES', 'AKSHAYA_TRITIYA_DAYS', 'BHAI_DOOJ_DAYS', 'BUDDHA_PURNIMA_DAYS', 'CHHATH_PUJA_DAYS', 'DHANTERAS_DAYS', 'DIWALI_DAYS', 'DUSSEHRA_(VIJAYADASHAMI)_DAYS', 'EID_UL_FITR_DAYS', 'GANESH_CHATURTHI_DAYS', 'GANGA_DUSSEHRA_DAYS', 'GOVARDHAN_POOJA_DAYS', 'GURU_PURNIMA_DAYS', 'HANUMAN_JAYANTI_DAYS', 'HARTALIK_TEEJ_DAYS', 'HOLI_DAYS', 'HOLIKA_DAHAN_DAYS', 'JAGANNATH_RATHYATRA_DAYS', 'JANMASHTAMI_DAYS', 'KARWA_CHAUTH_DAYS', 'LOHRI_DAYS', 'MAHA_SHIVARATRI_DAYS', 'MAKAR_SANKRANTI_PONGAL_DAYS', 'NAG_PANCHAMI_DAYS', 'NAVRATRI_DAYS', 'NEW_YEAR_DAYS', 'ONAM_DAYS', 'PITRAPA

In [16]:
#Separating the covariates
target_col = ["NET_SALES"]

#future covariates
future_covariates = [i for i in target_cols if i!='NET_SALES']

#actual_static_cols
actual_static_cols = [i for i in static_cols if i!='PARENT_DEALER_CODE_MODEL_FAMILY']

In [17]:
#since variables like MODEL_FAMILY,BRAKE_VARIANT,IGNITION_TYPE,WHEEL_TYPE,BIKE_COLOUR are mostly same for all the top 10 series, will be removing them from the static covariates'
static_covariates = [i for i in actual_static_cols if i not in ['MODEL_FAMILY','BRAKE_VARIANT','IGNITION_TYPE','WHEEL_TYPE','BIKE_COLOUR']]
static_covariates

['DEALER_CODE', 'DEALER_CITY', 'X_CITY_CATEGORY']

### Preparing data for Darts 

In [15]:
#Step 1 - Sorting the dataframe by date
pandas_df.reset_index().sort_values(by=["PARENT_DEALER_CODE_MODEL_FAMILY","MONTH_OF_SALE"]).set_index("MONTH_OF_SALE",inplace=True)

In [51]:
static_covariates

['DEALER_CODE', 'DEALER_CITY', 'X_CITY_CATEGORY']

In [18]:
#Step 2 - Separating the static covariates and NET_SALES column
pandas_df_with_target_and_static_covariates = pandas_df.loc[:,['PARENT_DEALER_CODE_MODEL_FAMILY','NET_SALES']+static_covariates]
pandas_df_with_target_and_static_covariates.head()

,PARENT_DEALER_CODE_MODEL_FAMILY,NET_SALES,DEALER_CODE,DEALER_CITY,X_CITY_CATEGORY
MONTH_OF_SALE,,,,,
2023-04-01,11736_SPLENDOR+_DRUM<>SELF<>CAST<>BLACK RED PU...,182.0,11736,KURUKSHETRA,URBAN
2023-04-01,10286_SPLENDOR+_DRUM<>SELF<>CAST<>BLACK RED PU...,762.0,10286,GANDHIDHAM,UNSPECIFIED
2023-04-01,11916_SPLENDOR+_DRUM<>SELF<>CAST<>BLACK AND AC...,151.0,11916,SANAND,URBAN
2023-04-01,11225_SPLENDOR+_DRUM<>SELF<>CAST<>BLACK RED PU...,366.0,11225,RAJKOT,UNSPECIFIED
2023-04-01,10841_SPLENDOR+_DRUM<>SELF<>CAST<>BLACK RED PU...,528.0,10841,BHAVNAGAR,UNSPECIFIED


In [43]:
#Step 3 - Separating the future covariates
pandas_df_with_future_covariates = pandas_df.loc[:,future_covariates]
pandas_df_with_future_covariates.head()

,NUM_FESTIVE_DAYS_MONTH,FESTIVE_PHASE_I,FESTIVE_PHASE_II,FESTIVE_PHASE_III,TOTAL_DAYS_FESTIVE_PHASE_I,TOTAL_DAYS_FESTIVE_PHASE_II,TOTAL_DAYS_FESTIVE_PHASE_III,TOTAL_DAYS_PITRU_PAKSH,PROP_FESTIVE_PHASE_I,PROP_EVENT_FESTIVE_PHASE_I,...,NAVRATRI_DAYS,NEW_YEAR_DAYS,ONAM_DAYS,PITRAPAKSHA_DAYS,RAKSHA_BANDHAN_DAYS,REPUBLIC_DAY_DAYS,VASANT_PANCHAMI_DAYS,VISHWAKARMA_PUJA_DAYS,MARRIAGE_DAYS,PARENT_DEALER_CODE_MODEL_FAMILY
MONTH_OF_SALE,,,,,,,,,,,,,,,,,,,,,
2023-04-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11736_SPLENDOR+_DRUM<>SELF<>CAST<>BLACK RED PU...
2023-04-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10286_SPLENDOR+_DRUM<>SELF<>CAST<>BLACK RED PU...
2023-04-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11916_SPLENDOR+_DRUM<>SELF<>CAST<>BLACK AND AC...
2023-04-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11225_SPLENDOR+_DRUM<>SELF<>CAST<>BLACK RED PU...
2023-04-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10841_SPLENDOR+_DRUM<>SELF<>CAST<>BLACK RED PU...


In [ ]:
#Step 4 - Creating the darts timeseries object for target and static covariates
darts_df_with_static_covariates = TimeSeries.from_group_dataframe(df=pandas_df_with_target_and_static_covariates,group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
                                                                  static_cols=static_covariates,value_cols=["NET_SALES"],freq='MS')


In [49]:
#Step 5 - Creating the darts timeseries object with future covariates

#Removing PARENT_DEALER_CODE_MODEL_FAMILY from future_covariates
future_covariates.remove('PARENT_DEALER_CODE_MODEL_FAMILY')

darts_df_with_future_covariates = TimeSeries.from_group_dataframe(df = pandas_df_with_future_covariates,
                                    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
                                    freq = 'MS',
                                    value_cols = future_covariates
                                    )

In [52]:
#Step 5 - Creating train, test, and validation split
#Train set - Apr'23 to Dec'25 
#Val set - Jan'26 to Mar'26 

#output_chunk_length will be 2

train_list = []
val_list = []
# test_list = []

for ts in darts_df_with_static_covariates:
    train,val = ts.split_after(pd.Timestamp('2025-12-01'))
    train_list.append(train)
    val_list.append(val)
    

In [ ]:
#Step 5 - Scaling
target_scaler = Scaler()
future_covariates_scaler = Scaler()


AttributeError: 'list' object has no attribute 'start_time'

In [ ]:
#Step 4 - Separate train, validation, and test datasets


train_data = 

target_scaler = Scaler()
series_scaled = target_scaler.fit_transform(darts_df_with_static_covariates)



In [8]:
#extract only NET_SALES
pandas_df_with_net_sales = pandas_df[["NET_SALES","PARENT_DEALER_CODE_MODEL_FAMILY"]]

#convert it into time series darts object
darts_df = TimeSeries.from_group_dataframe(df=pandas_df_with_net_sales,group_cols='PARENT_DEALER_CODE_MODEL_FAMILY',value_cols=["NET_SALES"],freq='MS')


In [9]:
# Function to encode the year as a normalized value
def encode_year(idx):
  return (idx.year - 2000) / 50

def encode_days_in_month(index):
  return index.days_in_month.to_numpy().reshape(-1,1)

# Set up the add_encoders dictionary to specify how different time-related encoders and transformers should be applied
add_encoders = {
    'cyclic': {'past': ['month'], 'future': ['month']},
    'position': {'past': ['relative'], 'future': ['relative']},
    'custom': {
        'past': [encode_year, encode_days_in_month],
        'future': [encode_year, encode_days_in_month]
    },
    'transformer': Scaler()
}

In [11]:
future_covariates

['NUM_FESTIVE_DAYS_MONTH',
 'FESTIVE_PHASE_I',
 'FESTIVE_PHASE_II',
 'FESTIVE_PHASE_III',
 'TOTAL_DAYS_FESTIVE_PHASE_I',
 'TOTAL_DAYS_FESTIVE_PHASE_II',
 'TOTAL_DAYS_FESTIVE_PHASE_III',
 'TOTAL_DAYS_PITRU_PAKSH',
 'PROP_FESTIVE_PHASE_I',
 'PROP_EVENT_FESTIVE_PHASE_I',
 'PROP_FESTIVE_PHASE_II',
 'PROP_EVENT_FESTIVE_PHASE_II',
 'PROP_FESTIVE_PHASE_III',
 'PROP_EVENT_FESTIVE_PHASE_III',
 'PROP_PITRU_PAKSH',
 'PROP_EVENT_PITRU_PAKSH',
 'AKSHAYA_TRITIYA_DAYS',
 'BHAI_DOOJ_DAYS',
 'BUDDHA_PURNIMA_DAYS',
 'CHHATH_PUJA_DAYS',
 'DHANTERAS_DAYS',
 'DIWALI_DAYS',
 'DUSSEHRA_(VIJAYADASHAMI)_DAYS',
 'EID_UL_FITR_DAYS',
 'GANESH_CHATURTHI_DAYS',
 'GANGA_DUSSEHRA_DAYS',
 'GOVARDHAN_POOJA_DAYS',
 'GURU_PURNIMA_DAYS',
 'HANUMAN_JAYANTI_DAYS',
 'HARTALIK_TEEJ_DAYS',
 'HOLI_DAYS',
 'HOLIKA_DAHAN_DAYS',
 'JAGANNATH_RATHYATRA_DAYS',
 'JANMASHTAMI_DAYS',
 'KARWA_CHAUTH_DAYS',
 'LOHRI_DAYS',
 'MAHA_SHIVARATRI_DAYS',
 'MAKAR_SANKRANTI_PONGAL_DAYS',
 'NAG_PANCHAMI_DAYS',
 'NAVRATRI_DAYS',
 'NEW_YEAR_DAYS',
 'O

In [12]:
actual_static_cols

['DEALER_CODE',
 'MODEL_FAMILY',
 'BRAKE_VARIANT',
 'IGNITION_TYPE',
 'WHEEL_TYPE',
 'BIKE_COLOUR',
 'DEALER_CITY',
 'X_CITY_CATEGORY']

['DEALER_CODE', 'DEALER_CITY', 'X_CITY_CATEGORY']

In [14]:
#Adding future covariates

holiday_cols = [i for i in future_covariates if i!='PARENT_DEALER_CODE_MODEL_FAMILY']

future_covariates_pandas_df = pandas_df.loc[:,future_covariates]

# #Convert it into timeseries dataframe
future_covariates_darts_df = TimeSeries.from_group_dataframe(
    df = future_covariates_pandas_df,
    freq = 'MS',
    value_cols = holiday_cols,
    group_cols = 'PARENT_DEALER_CODE_MODEL_FAMILY'
)
